In [ ]:
import xml.etree.ElementTree as ET
from tqdm import tqdm
import pickle

In [ ]:
path = './chess.stackexchange.com/'
# path = './math.stackexchange.com/'
# path = './crypto.stackexchange.com/'
# path = './physics.stackexchange.com/'

In [ ]:
tag_set = {}
print("Parse items...")
with open(path + 'Tags.xml', 'r') as f:
    # Skip the first two lines as in the original code
    for _ in range(2):
        next(f)
    
    # Read the rest of the lines one by one
    for line in tqdm(f):
        # if line.strip():  # To handle any empty lines if present
        if line == '</tags>': continue
        root = ET.fromstring(line)
        id = int(root.attrib['Id'])
        tag = root.attrib['TagName']
        tag_set[tag] = id

BI = {}
B = {}
notB = []
UB = {}
ans2ques = {}

tid = 200000

print("Parse bundles...")
with open(path + 'Posts.xml', 'r') as f:
    # Skip the first two lines
    for _ in range(2):
        next(f)

    # Process remaining lines one by one
    for line in tqdm(f):
        if line == '</posts>': continue
        root = ET.fromstring(line)
        id = int(root.attrib['Id'])
        ptype = int(root.attrib['PostTypeId'])

        if ptype == 1:
            rawtags = root.attrib['Tags']
            tags = rawtags.split('|')[1:-1]
            for t in tags:
                if t not in tag_set:
                    tag_set[t] = tid
                    tid += 1
            BI[id] = set([tag_set[t] for t in tags])
            sorted_integers = sorted(BI[id])
            B[id] = '|'.join(map(str, sorted_integers))

        if ptype > 2 or ptype == 0:
            notB.append(id)

notB = set(notB)

print("Parse answers...")
with open(path + 'Posts.xml', 'r') as f:
    # Skip the first two lines again
    for _ in range(2):
        next(f)

    # Process remaining lines one by one
    for line in tqdm(f):
        if line == '</posts>': continue
        root = ET.fromstring(line)
        id = int(root.attrib['Id'])
        ptype = int(root.attrib['PostTypeId'])

        if ptype == 2:
            pid = int(root.attrib['ParentId'])
            if pid in notB:
                continue
            ans2ques[id] = pid
            try:
                uid = int(root.attrib['OwnerUserId'])
            except:
                continue
            time = root.attrib['CreationDate']
            if uid not in UB:
                UB[uid] = []
            UB[uid].append((pid, time))
        

print("Parse comments...")
with open(path + 'Comments.xml', 'r') as f:
    # Skip the first two lines
    for _ in range(2):
        next(f)

    # Process remaining lines one by one
    for line in tqdm(f):
        if line == '</comments>': continue
        root = ET.fromstring(line)
        pid = int(root.attrib['PostId'])
        if pid in ans2ques:
            pid = ans2ques[pid]
        if pid in notB:
            continue
        try:
            uid = int(root.attrib['UserId'])
        except:
            continue
        time = root.attrib['CreationDate']
        if uid not in UB:
            UB[uid] = []
        UB[uid].append((pid, time))

insts = 0
for u in UB:
    insts += len(UB[u])

blen = 0
for b in BI:
    blen += len(BI[b])
blen /= len(BI)

print('Done.')
print('')

print(f'users: {len(UB)}')
print(f'bunds: {len(BI)}')
print(f'items: {len(tag_set)}')
print(f'insts: {insts}')
print(f'it/bd: {blen:.2f}')
print(f'in/us: {insts/len(UB):.2f}')

In [ ]:
def bundle_index(B):
    id = 1
    set2id = {}
    for b in tqdm(B):
        if B[b] not in set2id:
            set2id[B[b]] = id
            id += 1
    
    return set2id

def create_b2id(B, set2id):
    b2id = {}
    for b in B:
        b2id[b] = set2id[B[b]]
    return b2id

In [ ]:
set2id = bundle_index(B)
b2id = create_b2id(B, set2id)
for u in tqdm(UB):
    for t in range(len(UB[u])):
        try:
            b, timestamp = UB[u][t]
            b = b2id[b]
            UB[u][t] = (b, timestamp)
        except:
            print(u, t, UB[u][t])
newBI = {}
for b in tqdm(BI):
    bid = b2id[b]
    if bid not in newBI:
        newBI[bid] = BI[b]
BI = newBI

In [ ]:
k = 5

run = 0
while True:
    run += 1
    print(f'{run}th run...')
    u2del = []
    bcnt = {}
    for u in UB:
        if len(UB[u]) < k:
            u2del.append(u)
        for b, _ in UB[u]:
            if b not in bcnt:
                bcnt[b] = 0
            bcnt[b] += 1
    b2del = []
    for b in bcnt:
        if bcnt[b] < k:
            b2del.append(b)

    if len(u2del) + len(b2del) == 0:
        print('Finish.')
        break

    print(f'delete {len(u2del)} users and {len(b2del)} bundles...')

    for u in u2del:
        UB.pop(u)

    for b in b2del:
        BI.pop(b)
    
    b2del = set(b2del)
    for u in UB:
        UB[u] = [item for item in UB[u] if item[0] not in b2del]

b2del = []
for b in BI:
    if b not in bcnt:
        b2del.append(b)
for b in b2del:
    BI.pop(b)

In [ ]:
u2id = {}
b2id = {}
i2id = {}

uid = 1
bid = 1
iid = 1

for u in UB:
    u2id[u] = uid
    uid += 1
for b in BI:
    b2id[b] = bid
    bid += 1
    for i in BI[b]:
        if i not in i2id:
            i2id[i] = iid
            iid += 1

In [ ]:
for b in b2id:
    if b2id[b] == 2554: print(b)

In [ ]:
from datetime import datetime

def parse_time(tstr):
    dt = datetime.strptime(tstr, "%Y-%m-%dT%H:%M:%S.%f")
    return int(dt.timestamp())

In [ ]:
newUB = {}
newBI = {}
for u in UB:
    newUB[u2id[u]] = sorted([(b2id[b], parse_time(t)) for b, t in UB[u]], key = lambda x: x[1])
for b in BI:
    newBI[b2id[b]] = sorted([i2id[i] for i in BI[b]])

In [ ]:
inst = 0
with open(path+"user-bundle.txt", 'w') as f:
    for u in newUB:
        for b, t in newUB[u]:
            f.write(f'{u}\t{b}\t{t}\n')
            inst += 1
lensum = 0
with open(path+"bundle-item.txt", 'w') as f:
    for b in newBI:
        lensum += len(newBI[b])
        for i in newBI[b]:
            f.write(f'{b}\t{i}\n')

In [ ]:
print(f'user: {len(u2id)}')
print(f'bund: {len(b2id)}')
print(f'item: {len(i2id)}')
print(f'inst: {inst}')
print(f'blen: {lensum/len(b2id):.2f}')